# Les files — Exercice 4 : recherche et suppression des doublons

**Notions travaillées :** recherche dans une file, file auxiliaire, réutiliser ses propres fonctions.

## Mise en place

On importe la classe `File` écrite à l'exercice 1 (le fichier `file.py` doit être **dans le même dossier** que ce notebook).

L'énoncé interdit d'utiliser le type `list` : on ne manipule les files **qu'avec leurs méthodes** `enfile`, `defile`, `taille`, `est_vide`. La petite fonction `file_de` sert uniquement à fabriquer rapidement des files pour les tests.

On affiche une file sous la forme `tête < ... < queue` : on **défile à gauche** et on **enfile à droite**.

In [1]:
from file import File   # la classe File de l'exercice 1 (fichier file.py placé à côté du notebook)


def file_de(*elements):
    """Outil pour les tests : construit une file contenant les éléments donnés (le premier en tête)."""
    f = File()
    for el in elements:
        f.enfile(el)
    return f


print(file_de("a", "b", "c"))

tête < 'a' < 'b' < 'c' < queue


## Question 1 : `est_dans_la_file(f, val)`

C'est le même schéma que `nb_occurences` (exercice 2) : un **tour complet** de la file. On utilise un **booléen** `trouve`, qui passe à `True` dès qu'on rencontre `val`.

> **🔑 La technique clé de tout le TD : la rotation**
>
> Avec une file, on ne voit que la **tête**. Pour regarder tous les éléments, on fait faire **un tour complet** à la file :
> on défile un élément, on le traite, puis on le **ré-enfile**. Après exactement `f.taille()` rotations, chaque élément est revenu à sa place : la file est **identique** à ce qu'elle était au départ.
>
> ```python
> for _ in range(f.taille()):
>     el = f.defile()
>     ...            # traiter el
>     f.enfile(el)
> ```
>
> ⚠️ On calcule `f.taille()` **une seule fois**, au départ (c'est ce que fait `range`). Une boucle `while not f.est_vide()` ne s'arrêterait jamais, puisqu'on ré-enfile chaque élément.


⚠️ **Le piège de cet exercice :** avec une liste, on écrirait `return True` dès qu'on trouve la valeur. Ici, ce serait une **erreur** : la fonction s'arrêterait au milieu du tour, et les éléments déjà défilés se retrouveraient **derrière** les autres. La file serait modifiée ! Il faut **terminer le tour** avant de répondre.

In [2]:
def est_dans_la_file(f, val):
    trouve = False
    for _ in range(f.taille()):
        el = f.defile()
        if el == val:
            trouve = True
        f.enfile(el)             # on termine toujours le tour complet
    return trouve


f = file_de(5, 8, 2, 8)
print(est_dans_la_file(f, 2), est_dans_la_file(f, 7))
print("La file après les appels :", f)

assert est_dans_la_file(f, 2)
assert not est_dans_la_file(f, 7)
assert repr(f) == repr(file_de(5, 8, 2, 8))   # file inchangée
assert not est_dans_la_file(File(), 1)

True False
La file après les appels : tête < 5 < 8 < 2 < 8 < queue


Démonstration du piège : avec un `return True` dans la boucle, la file est abîmée.

In [3]:
def est_dans_la_file_FAUX(f, val):
    for _ in range(f.taille()):
        el = f.defile()
        f.enfile(el)
        if el == val:
            return True         # on sort au milieu du tour !
    return False


f = file_de(5, 8, 2, 8)
est_dans_la_file_FAUX(f, 8)
print(f, " ← l'ordre a changé : 5 est passé derrière")

tête < 2 < 8 < 5 < 8 < queue  ← l'ordre a changé : 5 est passé derrière


## Question 2 : `supprime_doublons(f)`

On veut garder **une seule fois** chaque valeur, en gardant la **première apparition** et l'ordre : `3 1 3 2 1 3` → `3 1 2`.

**Difficulté :** pour savoir si un élément est un doublon, il faut se souvenir des valeurs **déjà rencontrées**. On n'a pas le droit aux listes… mais on a le droit aux **files** ! On utilise une file auxiliaire `vues`.

**Algorithme :** on fait un tour complet de `f`. Pour chaque élément défilé :
- s'il est **déjà dans** `vues` (on réutilise `est_dans_la_file` de la question 1), c'est un doublon : on ne le ré-enfile **pas**, il disparaît ;
- sinon, on le note dans `vues` et on le **ré-enfile** dans `f`.

| Élément défilé | Déjà vu ? | `vues` | `f` (partie déjà traitée) |
|:---|:---|:---|:---|
| 3 | non | `3` | `3` |
| 1 | non | `3 1` | `3 1` |
| 3 | **oui** → supprimé | `3 1` | `3 1` |
| 2 | non | `3 1 2` | `3 1 2` |
| 1 | **oui** → supprimé | `3 1 2` | `3 1 2` |
| 3 | **oui** → supprimé | `3 1 2` | `3 1 2` |

💡 La modification est bien **en place** : à la fin, c'est `f` elle-même qui ne contient plus de doublons (la file `vues` n'est qu'un outil de travail).

In [4]:
def supprime_doublons(f):
    vues = File()
    for _ in range(f.taille()):
        el = f.defile()
        if not est_dans_la_file(vues, el):
            vues.enfile(el)
            f.enfile(el)
        # sinon : doublon, on ne le remet pas


f = file_de(3, 1, 3, 2, 1, 3)
supprime_doublons(f)
print(f)
assert repr(f) == repr(file_de(3, 1, 2))

g = file_de("a", "a", "a")
supprime_doublons(g)
assert repr(g) == repr(file_de("a"))
h = File()
supprime_doublons(h)
assert h.est_vide()
print("Tous les tests passent ✔")

tête < 3 < 1 < 2 < queue
Tous les tests passent ✔


💡 **Coût :** pour chacun des `n` éléments, on parcourt la file `vues`, qui contient jusqu'à `n` éléments : le coût est de l'ordre de `n²` (coût **quadratique**). Avec un ensemble python (`set`), on ferait mieux, mais l'énoncé impose de n'utiliser que des files.